# Combined TI × LI × SI Pipeline (aligned with final_wingx + wingx-smid-kmeans)

**Merges Time-of-Day Index (TI), Location Index (LI), and Seasonality Index (SI) with raw indices and multipliers**

**Input**: `lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet`  
**Output**: `combined_ti_li_si_output.csv`

**Final schema**: Full audit trail with raw indices AND multipliers per CTS framework

**Scope**: 16 priority corridors (6 Light Jet + 10 Super Midsize Jet) × actual months × 7 DOW × 6 TOD (data-driven, not synthetic)

**Key alignment**: Hours filters (LJ ≥1.5, SMID ≥2.5) matching final_wingx and wingx-smid-kmeans notebooks

## Configuration

In [3]:
import pandas as pd
import numpy as np

# ─── CONFIGURATION REFERENCE ────────────────────────────────────────────────────
# To match different sources, use these presets:
#
# OPTION A: Comprehensive Market (RECOMMENDED — CURRENT)
#   → 16 LJ models + 2024-2025 years
#   → Covers full LJ market (100%), recent data patterns
#   → SI values will differ from wingx-lj-kmeans (different data volume)
#   → Best for: Forward pricing across full LJ segment
#
# OPTION B: Replicate wingx-lj-kmeans exactly
#   → 6 partial strings: ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']
#   → SELECTED_YEARS = None (all years 2023-2025)
#   → Corridor format: " ➔ " (arrow character, not ->)
#   → SI values match wingx-lj-kmeans exactly (102k missions, Phenom-only)
#   → Best for: Matching wingx-lj-kmeans SI values exactly
#
# OPTION C: final_wingx style (16 models + 2024-2025)
#   → Same as Option A (this is Option A)
#
# Switch: Update LIGHT_JET_MODELS and SELECTED_YEARS below
# ────────────────────────────────────────────────────────────────────────────────

# Input file (local)
RAW_PARQUET = 'gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_local_TimeZone.parquet'

# ─── DATE RANGE FILTERING (OPTIONAL) ────────────────────────────────────────────
# Set both to None for NO date filtering (use SELECTED_YEARS instead)
# OR set both to filter by date range (ignores SELECTED_YEARS if date range is set)
# Format: 'YYYY-MM-DD'
#
# Examples:
# START_DATE = '2024-05-01'  # May 1, 2024
# END_DATE = '2025-06-30'    # June 30, 2025
#
# START_DATE = None  # No date filtering
# END_DATE = None
#
START_DATE = None  # Change to '2024-05-01' for May 1 start
END_DATE = None    # Change to '2025-06-30' for June 30 end

# Time periods (OPTION A: Comprehensive Market)
SELECTED_YEARS = [2024, 2025]  # Recent 2 years (Option A) — ignored if START_DATE/END_DATE set
# To match wingx-lj-kmeans: SELECTED_YEARS = None  # All years (Option B)

MEAN_CELL_SHARE = 1 / 42  # 6 TOD × 7 DOW = 42 cells per corridor

# Filtering thresholds
MIN_MISSIONS_LJ = 20      # LJ: min annual missions per corridor
MIN_MISSIONS_SMID = 30    # SMID: min annual missions per corridor
MIN_HOURS_LJ = 1.5        # Hours filter for Light Jet
MIN_HOURS_SMID = 2.5      # Hours filter for SMID

# Aircraft models (OPTION A: Comprehensive Market — 16 exact model names)
LIGHT_JET_MODELS = [
    'Phenom 300', 'Phenom 300-E', 
    'Citation CJ3', 'Citation CJ3+',
    'Citation CJ4', 'Hawker 400XP', 
    'Citation CJ2+', 'Citation Ultra',
    'Citation V', '400-A', 
    'Citation CJ4 Gen2', 'Citation Encore+'
]
# ^ OPTION A: 16 exact models (full market ~323k missions)
# To match wingx-lj-kmeans (Option B), replace with:
# LIGHT_JET_MODELS = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra']

WU_OPERATORS = [
    'Wheels Up Private Jets', 'Wheels Up LLC',
    'Mountain Aviation', 'Alante Air Charter'
]

# Time categories
DAY_ORDER    = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
BUCKET_ORDER = ['00:00-06:59', '07:00-08:59', '09:00-12:59',
                '13:00-15:59', '16:00-18:59', '19:00-23:59']
MONTH_NAMES  = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# TI Score mapping (ratio_vs_mean → slab → score)
TI_SCORE_MAP = {
    "> 0 to <= 0.5x":      1,
    "> 0.5x to 0.6x":      2,
    "> 0.6x to 0.75x":     3,
    "> 0.75x to 1.0x":     4,
    "> 1.0x to 1.1x":      5,
    "> 1.1x to 1.25x":     6,
    "> 1.25x to 1.5x":     7,
    "> 1.5x to 1.75x":     8,
    "> 1.75x to 2.0x":     9,
    "> 2.0x to 3.0x":      10,
    "> 3.0x":              10,
}
TI_BINS   = [0, 0.5, 0.6, 0.75, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0, 3.0, float('inf')]
TI_LABELS = list(TI_SCORE_MAP.keys())

# SI Multiplier mapping (seasonality_index → slab → multiplier)
SI_MULTIPLIER_MAP = {
    "< 0.75x":            0.75,
    "0.75x to < 0.90x":   0.90,
    "0.90x to < 1.10x":   1.00,
    "1.10x to < 1.30x":   1.15,
    "1.30x to < 1.60x":   1.30,
    ">= 1.60x":           1.50,
}
SI_BINS   = [0, 0.75, 0.90, 1.10, 1.30, 1.60, float('inf')]
SI_LABELS = list(SI_MULTIPLIER_MAP.keys())

# LI Multiplier mapping (LI_index → slab → multiplier)
LI_MULTIPLIER_MAP = {
    "<= 0.75x":           0.85,
    "> 0.75x to 1.00x":   0.95,
    "> 1.00x to 1.50x":   1.00,
    "> 1.50x to 2.50x":   1.10,
    "> 2.50x to 4.00x":   1.20,
    "> 4.00x":            1.30,
}
LI_BINS   = [0, 0.75, 1.00, 1.50, 2.50, 4.00, float('inf')]
LI_LABELS = list(LI_MULTIPLIER_MAP.keys())

# ─── CTS TOGGLE MAP (CONFIGURABLE - TUPLE FORMAT) ────────────────────────────────
# Easy to edit: (min, max, toggle%) — copy/paste from Excel!
# Format: {segment: [(min, max, toggle%), ...]}
# Add/edit/remove rows easily — just modify the tuples

CTS_TOGGLE_MAP = {
    'Light Jet': [
        (0, 0.25, -15),     (0.25, 0.5, -15),    (0.5, 0.75, -15),    (0.75, 1, -15),
        (1, 1.25, -10),     (1.25, 1.5, -10),    (1.5, 1.75, -10),    (1.75, 2, -10),
        (2, 2.25, -10),     (2.25, 2.5, -5),     (2.5, 2.75, -5),     (2.75, 3, -5),
        (3, 3.25, -5),      (3.25, 3.5, 0),      (3.5, 3.75, 0),      (3.75, 4, 0),
        (4, 4.25, 0),       (4.25, 4.5, 0),      (4.5, 4.75, 0),      (4.75, 5, 0),
        (5, 5.25, 0),       (5.25, 5.5, 5),      (5.5, 5.75, 5),      (5.75, 6, 5),
        (6, 6.25, 5),       (6.25, 6.5, 10),     (6.5, 6.75, 10),     (6.75, 7, 10),
        (7, 7.25, 15),      (7.25, 7.5, 15),     (7.5, 7.75, 15),     (7.75, 8, 15),
        (8, 8.25, 20),      (8.25, 8.5, 20),     (8.5, 8.75, 20),     (8.75, 9, 25),
        (9, 9.25, 25),      (9.25, 9.5, 25),     (9.5, 9.75, 30),     (9.75, 10, 30),
        (10, float('inf'), 30),
    ],
    'Super Midsize Jet': [
        (0, 0.25, -15),     (0.25, 0.5, -10),    (0.5, 0.75, -5),     (0.75, 1, -5),
        (1, 1.25, -5),      (1.25, 1.5, -5),     (1.5, 1.75, 0),      (1.75, 2, 0),
        (2, 2.25, 0),       (2.25, 2.5, 0),      (2.5, 2.75, 0),      (2.75, 3, 0),
        (3, 3.25, 0),       (3.25, 3.5, 0),      (3.5, 3.75, 0),      (3.75, 4, 5),
        (4, 4.25, 5),       (4.25, 4.5, 5),      (4.5, 4.75, 5),      (4.75, 5, 5),
        (5, 5.25, 5),       (5.25, 5.5, 5),      (5.5, 5.75, 10),     (5.75, 6, 10),
        (6, 6.25, 10),      (6.25, 6.5, 10),     (6.5, 6.75, 10),     (6.75, 7, 10),
        (7, 7.25, 10),      (7.25, 7.5, 10),     (7.5, 7.75, 10),     (7.75, 8, 10),
        (8, 8.25, 10),      (8.25, 8.5, 10),     (8.5, 8.75, 10),     (8.75, 9, 10),
        (9, 9.25, 15),      (9.25, 9.5, 15),     (9.5, 9.75, 15),     (9.75, 10, 15),
        (10, float('inf'), 15),
    ]
}

# Priority corridors
LJ_CORRIDORS = [
    'Denver->LA Basin',           'LA Basin->Denver',
    'Charlotte->South Florida',   'South Florida->Charlotte',
    'Chicago->South Florida',     'South Florida->Chicago',
]

SMID_CORRIDORS = [
    'New York City->South Florida', 'South Florida->New York City',
    'Denver->South Florida',        'South Florida->Denver',
    'Chicago->South Florida',       'South Florida->Chicago',
    'Phoenix Valley->Boston',       'Boston->Phoenix Valley',
    'LA Basin->New York City',      'New York City->LA Basin',
]

print("✓ Configuration loaded (OPTION A: Comprehensive Market)")
print(f"  LJ models: {len(LIGHT_JET_MODELS)} (full market coverage)")
if START_DATE and END_DATE:
    print(f"  Date range: {START_DATE} to {END_DATE}")
else:
    print(f"  Year range: {SELECTED_YEARS}")
print(f"  LJ hours: >= {MIN_HOURS_LJ} | min_missions > {MIN_MISSIONS_LJ}")
print(f"  SMID hours: >= {MIN_HOURS_SMID} | min_missions > {MIN_MISSIONS_SMID}")
print(f"  CTS Toggle map loaded: {len(CTS_TOGGLE_MAP)} segments")
print(f"\n📍 To edit date range filtering:")
print(f"  Change START_DATE and END_DATE in Cell 2 (format: 'YYYY-MM-DD')")
print(f"  Example: START_DATE = '2024-05-01'  (May 1, 2024)")
print(f"  Example: END_DATE = '2025-06-30'    (June 30, 2025)")
print(f"\n📍 To edit CTS toggle mapping (easy!):")
print(f"  Edit CTS_TOGGLE_MAP in Cell 2")
print(f"  Change tuples: (min, max, toggle%)")
print(f"  Example: (5, 5.25, 0) → (5, 5.25, 5)  [increase toggle from 0% to 5%]")
print(f"\n  Light Jet ranges: {len(CTS_TOGGLE_MAP['Light Jet'])} tuples")
print(f"  Super Midsize Jet ranges: {len(CTS_TOGGLE_MAP['Super Midsize Jet'])} tuples")

✓ Configuration loaded (OPTION A: Comprehensive Market)
  LJ models: 12 (full market coverage)
  Year range: [2024, 2025]
  LJ hours: >= 1.5 | min_missions > 20
  SMID hours: >= 2.5 | min_missions > 30
  CTS Toggle map loaded: 2 segments

📍 To edit date range filtering:
  Change START_DATE and END_DATE in Cell 2 (format: 'YYYY-MM-DD')
  Example: START_DATE = '2024-05-01'  (May 1, 2024)
  Example: END_DATE = '2025-06-30'    (June 30, 2025)

📍 To edit CTS toggle mapping (easy!):
  Edit CTS_TOGGLE_MAP in Cell 2
  Change tuples: (min, max, toggle%)
  Example: (5, 5.25, 0) → (5, 5.25, 5)  [increase toggle from 0% to 5%]

  Light Jet ranges: 41 tuples
  Super Midsize Jet ranges: 41 tuples


## Step 1: Load & Preprocess

In [4]:
df = pd.read_parquet(RAW_PARQUET)
df['FlightDate_local'] = pd.to_datetime(df['FlightDate_local'])
df['year_local']  = df['FlightDate_local'].dt.year
df['month_local'] = df['FlightDate_local'].dt.month
df = df.rename(columns={'FromCluster': 'FromMetro', 'ToCluster': 'ToMetro'})

print(f"✓ Loaded {len(df):,} records (all years)")
print(f"  Date range: {df['FlightDate_local'].min().date()} to {df['FlightDate_local'].max().date()}")

# ─── APPLY DATE RANGE FILTERING (if configured) ────────────────────────────────
if START_DATE and END_DATE:
    start = pd.to_datetime(START_DATE)
    end = pd.to_datetime(END_DATE)
    df = df[(df['FlightDate_local'] >= start) & (df['FlightDate_local'] <= end)]
    print(f"\n✓ Date range filter applied: {START_DATE} to {END_DATE}")
    print(f"  Records after filter: {len(df):,}")
    print(f"  Actual date range: {df['FlightDate_local'].min().date()} to {df['FlightDate_local'].max().date()}")

# ─── DATA FOR TI/LI: FILTERED BY SELECTED_YEARS (or date range above) ────────────
if SELECTED_YEARS is not None and not (START_DATE and END_DATE):
    # Use SELECTED_YEARS filter only if date range not set
    data = df[
        df['year_local'].isin(SELECTED_YEARS) &
        (df['FromMetro'] != df['ToMetro']) &
        (df['FromMetro'] != 'OTHER_METRO') &
        (df['ToMetro'] != 'OTHER_METRO')
    ].copy()
    print(f"✓ TI/LI data filtered to {SELECTED_YEARS}")
else:
    # Use date range (already filtered above) or all data
    data = df[
        (df['FromMetro'] != df['ToMetro']) &
        (df['FromMetro'] != 'OTHER_METRO') &
        (df['ToMetro'] != 'OTHER_METRO')
    ].copy()
    if START_DATE and END_DATE:
        print(f"✓ TI/LI data: date range {START_DATE} to {END_DATE}")
    else:
        print(f"✓ TI/LI data: no year/date filter applied (using all data)")

# ─── DATA FOR SI: ALL DATA (use same date range as TI/LI if set) ─────────────────
si_data_raw = df[
    (df['FromMetro'] != df['ToMetro']) &
    (df['FromMetro'] != 'OTHER_METRO') &
    (df['ToMetro'] != 'OTHER_METRO')
].copy()
print(f"✓ SI data: all data {si_data_raw['year_local'].min():.0f}-{si_data_raw['year_local'].max():.0f} ({len(si_data_raw):,} records)")

# Build corridors and flags for both datasets
data['Route']   = data['FromMetro'] + '->' + data['ToMetro']
data['corridor_city_mapping'] = data['FromCity'] + '->' + data['ToCity']
data['is_wu']   = data['Operator'].isin(WU_OPERATORS)

si_data_raw['Route'] = si_data_raw['FromMetro'] + '->' + si_data_raw['ToMetro']
si_data_raw['corridor_city_mapping'] = si_data_raw['FromCity'] + '->' + si_data_raw['ToCity']
si_data_raw['is_wu'] = si_data_raw['Operator'].isin(WU_OPERATORS)

# Categorize days and time buckets
data['day_name']    = pd.Categorical(data['dow_local'],         categories=DAY_ORDER,    ordered=True)
data['hour_bucket'] = pd.Categorical(data['hour_bucket_local'], categories=BUCKET_ORDER, ordered=True)

si_data_raw['day_name']    = pd.Categorical(si_data_raw['dow_local'],         categories=DAY_ORDER,    ordered=True)
si_data_raw['hour_bucket'] = pd.Categorical(si_data_raw['hour_bucket_local'], categories=BUCKET_ORDER, ordered=True)

print(f"✓ Preprocessed TI/LI data: {len(data):,} inter-metro records")
print(f"✓ Preprocessed SI data: {len(si_data_raw):,} inter-metro records")

✓ Loaded 2,140,706 records (all years)
  Date range: 2023-12-31 to 2025-12-31
✓ TI/LI data filtered to [2024, 2025]
✓ SI data: all data 2023-2025 (1,795,455 records)
✓ Preprocessed TI/LI data: 1,795,360 inter-metro records
✓ Preprocessed SI data: 1,795,455 inter-metro records


In [22]:
data[data["Route"]=="Denver->LA Basin"]["corridor_city_mapping"].unique()

array(['EAGLE->San Diego', 'RIFLE->PALM SPRINGS', 'TAOS->SANTA BARBARA',
       ..., 'Colby->Las Vegas', 'Laramie->ONTARIO', 'Burley->San Diego'],
      shape=(1142,), dtype=object)

## Step 2: Segment Split

In [5]:
smid_data = data[data['aircraft_segment'] == 'Super Midsize Jet'].copy()
lj_data   = data[data['aircraft_model'].isin(LIGHT_JET_MODELS)].copy()

print(f"✓ TI/LI DATA (recent, focused):")
print(f"  LJ records: {len(lj_data):,}")
print(f"  LJ models in data: {lj_data['aircraft_model'].nunique()}/{len(LIGHT_JET_MODELS)} configured")
print(f"✓ SMID records: {len(smid_data):,}")

# ─── SI DATA: ALL MODELS, ALL YEARS ─────────────────────────────────────────
smid_si_data = si_data_raw[si_data_raw['aircraft_segment'] == 'Super Midsize Jet'].copy()
lj_si_data   = si_data_raw[si_data_raw['aircraft_segment'] == 'Light Jet'].copy()  # All LJ models, all years

print(f"\n✓ SI DATA (all models, all years):")
print(f"  LJ records: {len(lj_si_data):,} ({lj_si_data['aircraft_model'].nunique()} unique models)")
print(f"  SMID records: {len(smid_si_data):,}")
print(f"  Year range: {si_data_raw['year_local'].min():.0f}-{si_data_raw['year_local'].max():.0f}")

# ─── BUILD ROUTE→CORRIDOR MAPPING (OPTIMIZED: Keep Together Throughout) ────
# Create mapping once, pass to all compute functions
# Use most-frequent city pair per route as the representative corridor
corridor_map = (
    data.groupby(['Route', 'corridor_city_mapping'])
    .size()
    .reset_index(name='_count')
    .sort_values('_count', ascending=False)
    .drop_duplicates(subset='Route', keep='first')
    [['Route', 'corridor_city_mapping']]
    .rename(columns={'corridor_city_mapping': 'Corridor'})
)

print(f"\n✓ Route→Corridor mapping: {len(corridor_map)} routes (one representative city pair per route)")
print(f"  Example mappings:")
for idx, (route, corr) in enumerate(corridor_map.head(3).values):
    print(f"    {route} → {corr}")

# Also create mapping for SI data (same routes, same mapping)
si_corridor_map = corridor_map.copy()
print(f"\n✓ Mapping will be passed to all TI/LI/SI functions for consistency")

✓ TI/LI DATA (recent, focused):
  LJ records: 594,117
  LJ models in data: 12/12 configured
✓ SMID records: 916,869

✓ SI DATA (all models, all years):
  LJ records: 878,547 (45 unique models)
  SMID records: 916,908
  Year range: 2023-2025

✓ Route→Corridor mapping: 462 routes (one representative city pair per route)
  Example mappings:
    Houston->Dallas → Houston->Dallas
    Dallas->Houston → Dallas->Houston
    Dallas->San Antonio → Dallas->Austin

✓ Mapping will be passed to all TI/LI/SI functions for consistency


# ─── MODEL AUDIT: Find ALL distinct LJ models in dataset (like final_wingx Cell 1.5) ────


In [6]:

# 1. Get ALL Light Jet records (before any model filtering)
df_lj_all = data[data['aircraft_segment'] == 'Light Jet'].copy()

# 2. Get distinct models and their mission counts
lj_all_models = df_lj_all['aircraft_model'].value_counts().reset_index()
lj_all_models.columns = ['aircraft_model', 'mission_count']

print(f"\n📋 DISTINCT LIGHT JET MODELS IN DATASET ({len(lj_all_models)} found):")
print("─" * 70)
display(lj_all_models)

# 3. Show which models from your config are in the dataset (EXACT MATCH)
print(f"\n🔍 CONFIGURATION MODEL MATCHING (EXACT):")
print(f"Config has: {LIGHT_JET_MODELS}")
print(f"\nExact matches in dataset:")
for model in LIGHT_JET_MODELS:
    matches = lj_all_models[lj_all_models['aircraft_model'] == model]
    if not matches.empty:
        count = matches['mission_count'].values[0]
        pct = (count / len(df_lj_all)) * 100
        print(f"  ✓ '{model}': {count:,} missions ({pct:.1f}%)")
    else:
        print(f"  ✗ '{model}': NOT FOUND in dataset")

# 4. Pattern matching (like final_wingx) - show what REGEX patterns would match
print(f"\n🔍 PATTERN MATCHING (SUBSTRING/REGEX - like final_wingx):")
pattern_list = ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra', '400']
print(f"Testing patterns: {pattern_list}\n")

for pattern in pattern_list:
    matches = lj_all_models[lj_all_models['aircraft_model'].str.contains(pattern, case=False, na=False)]
    if not matches.empty:
        model_list = matches['aircraft_model'].tolist()
        total_missions = matches['mission_count'].sum()
        print(f"  '{pattern}' matches:")
        for model in model_list:
            count = matches[matches['aircraft_model'] == model]['mission_count'].values[0]
            print(f"    • {model}: {count:,} missions")
    else:
        print(f"  '{pattern}': NO MATCHES")

# 5. Summary stats
config_coverage = df_lj_all[df_lj_all['aircraft_model'].isin(LIGHT_JET_MODELS)]
print(f"\n📊 COVERAGE SUMMARY:")
print(f"  Total LJ missions (all models): {len(df_lj_all):,}")
print(f"  LJ missions matching config: {len(config_coverage):,}")
print(f"  Coverage: {(len(config_coverage) / len(df_lj_all) * 100):.1f}%")
print(f"  Unique models in config: {len([m for m in LIGHT_JET_MODELS if m in lj_all_models['aircraft_model'].values])}")
print(f"  Unique models NOT in config: {len(lj_all_models) - len([m for m in LIGHT_JET_MODELS if m in lj_all_models['aircraft_model'].values])}")


📋 DISTINCT LIGHT JET MODELS IN DATASET (45 found):
──────────────────────────────────────────────────────────────────────


,aircraft_model,mission_count
0,Phenom 300,222970
1,Citation CJ3,78148
2,no model available,70921
3,Phenom 300-E,53569
4,Citation CJ3+,47725
5,Citation CJ4,42190
6,400-A,31204
7,Citation Ultra,30196
8,Citation V,28625
9,400-XP,27103



🔍 CONFIGURATION MODEL MATCHING (EXACT):
Config has: ['Phenom 300', 'Phenom 300-E', 'Citation CJ3', 'Citation CJ3+', 'Citation CJ4', 'Hawker 400XP', 'Citation CJ2+', 'Citation Ultra', 'Citation V', '400-A', 'Citation CJ4 Gen2', 'Citation Encore+']

Exact matches in dataset:
  ✓ 'Phenom 300': 222,970 missions (25.4%)
  ✓ 'Phenom 300-E': 53,569 missions (6.1%)
  ✓ 'Citation CJ3': 78,148 missions (8.9%)
  ✓ 'Citation CJ3+': 47,725 missions (5.4%)
  ✓ 'Citation CJ4': 42,190 missions (4.8%)
  ✓ 'Hawker 400XP': 21,675 missions (2.5%)
  ✓ 'Citation CJ2+': 16,210 missions (1.8%)
  ✓ 'Citation Ultra': 30,196 missions (3.4%)
  ✓ 'Citation V': 28,625 missions (3.3%)
  ✓ '400-A': 31,204 missions (3.6%)
  ✓ 'Citation CJ4 Gen2': 13,281 missions (1.5%)
  ✓ 'Citation Encore+': 8,324 missions (0.9%)

🔍 PATTERN MATCHING (SUBSTRING/REGEX - like final_wingx):
Testing patterns: ['Phenom 300', 'Citation CJ', 'Hawker 400', 'Beechjet', 'Encore', 'Ultra', '400']

  'Phenom 300' matches:
    • Phenom 300: 222,9

In [7]:
def compute_si(df, corridor_map, corridors, cabin_label, min_hours, min_missions):
    """
    Compute Seasonality Index multiplier per corridor × month.

    Uses all historical data (all models, all years) for robust seasonality patterns.
    Includes Route + Corridor mapping from the start (optimized).

    Steps:
    1. Filter by min_hours
    2. Merge corridor_map to add Corridor dimension
    3. Group by corridor × month
    4. Filter to corridors with > min_missions annual missions
    5. Compute monthly_density = monthly_flights / annual_flights
    6. Compute SI = monthly_density / (1/12)
    7. Bin to SI_multiplier

    Returns: Route, Corridor, cabin, month, month_num, monthly_density, seasonality_index, si_slab, SI_multiplier
    """
    # Filter by hours AND corridor list
    df_corr = df[(df['Route'].isin(corridors)) & (df['Hours'] >= min_hours)].copy()
    
    # ✅ OPTIMIZED: Merge corridor_map early to keep Route + Corridor together
    df_corr = df_corr.merge(corridor_map[['Route', 'Corridor']], on='Route', how='left')

    # Step 1: Monthly flight counts by Route + Corridor
    monthly = df_corr.groupby(['Route','Corridor','month_local']).size().unstack(fill_value=0)
    monthly = monthly.reindex(columns=range(1,13), fill_value=0)

    # Step 2: FILTER - min_missions threshold
    annual = monthly.sum(axis=1)
    mask_min_missions = annual > min_missions
    monthly_filtered = monthly[mask_min_missions].copy()
    annual_filtered = annual[mask_min_missions].copy()

    if len(monthly_filtered) == 0:
        print(f"  ⚠️  [{cabin_label}] No corridors passed filters (hours >= {min_hours} AND missions > {min_missions})")
        return pd.DataFrame()

    print(f"  ✓ [{cabin_label}] {len(monthly_filtered)} corridors (ALL models, ALL years)")

    # Step 3: Normalize to monthly density
    density = monthly_filtered.div(annual_filtered, axis=0)

    # Step 4: Create seasonality index
    si = density / (1/12)

    # Convert to long format (preserves Route, Corridor from MultiIndex)
    si_long = si.stack().rename('seasonality_index').reset_index()
    si_long.columns = ['Route','Corridor','month_num','seasonality_index']

    # Add monthly_density
    density_long = density.stack().rename('monthly_density').reset_index()
    density_long.columns = ['Route','Corridor','month_num','monthly_density']
    
    si_long = si_long.merge(density_long[['Route', 'Corridor', 'month_num', 'monthly_density']], 
                             on=['Route', 'Corridor', 'month_num'])

    # Bin into SI slab label
    si_long['si_slab'] = pd.cut(
        si_long['seasonality_index'],
        bins=SI_BINS,
        labels=SI_LABELS,
        right=False
    )

    # Map slab to multiplier
    si_long['SI_multiplier'] = si_long['si_slab'].map(SI_MULTIPLIER_MAP).astype(float)

    # Add month name
    si_long['month'] = si_long['month_num'].map(MONTH_NAMES)
    si_long['cabin'] = cabin_label

    # ✅ Return with Route + Corridor together
    return si_long[['Route','Corridor','cabin','month','month_num','monthly_density','seasonality_index','si_slab','SI_multiplier']]

print("\n🔄 Computing SI (using ALL models, ALL years for robust seasonality):")
print(f"  LJ: ALL models, all years, min_missions > {MIN_MISSIONS_LJ}")
print(f"  SMID: ALL years, min_missions > {MIN_MISSIONS_SMID}")
lj_si   = compute_si(lj_si_data, si_corridor_map, LJ_CORRIDORS,   'Light Jet',        MIN_HOURS_LJ,   MIN_MISSIONS_LJ)
smid_si = compute_si(smid_si_data, si_corridor_map, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID, MIN_MISSIONS_SMID)

print(f"\n✓ LJ SI: {len(lj_si)} rows")
print(f"✓ SMID SI: {len(smid_si)} rows")


🔄 Computing SI (using ALL models, ALL years for robust seasonality):
  LJ: ALL models, all years, min_missions > 20
  SMID: ALL years, min_missions > 30
  ✓ [Light Jet] 6 corridors (ALL models, ALL years)
  ✓ [Super Midsize Jet] 10 corridors (ALL models, ALL years)

✓ LJ SI: 72 rows
✓ SMID SI: 120 rows


## Step 4: TI Computation (with hours filter)

In [8]:
def compute_ti(df, corridor_map, corridors, cabin_label, min_hours):
    """
    Compute TI Score (0-10) per corridor × day × hour_bucket.
    Includes Route + Corridor mapping from the start (optimized).
    Includes hours filter before groupby.
    
    Returns: Route, Corridor, cabin, DOW, TOD, cell_share, ratio_vs_mean, ti_slab, TI_score
    """
    df_corr = df[(df['Route'].isin(corridors)) & (df['Hours'] >= min_hours)].copy()
    
    # ✅ OPTIMIZED: Merge corridor_map early to keep Route + Corridor together
    df_corr = df_corr.merge(corridor_map[['Route', 'Corridor']], on='Route', how='left')

    # Count flights in each (Route, Corridor, day, time) cell
    grouped = (
        df_corr.groupby(['Route','Corridor','day_name','hour_bucket'], observed=False)
        .size()
        .reset_index(name='flight_count')
    )

    # Total flights per corridor (across all cells)
    grouped['total_corridor_flights'] = grouped.groupby(['Route','Corridor'])['flight_count'].transform('sum')

    # Cell share (what fraction of corridor flights are in this cell)
    grouped['cell_share'] = grouped['flight_count'] / grouped['total_corridor_flights']

    # Ratio vs mean (how much above/below flat 1/42)
    grouped['ratio_vs_mean'] = grouped['cell_share'] / MEAN_CELL_SHARE

    # Bin into TI slab label (right=True for exclusive left, inclusive right)
    grouped['ti_slab'] = pd.cut(
        grouped['ratio_vs_mean'],
        bins=TI_BINS,
        labels=TI_LABELS,
        right=True,
        ordered=False
    )

    # Map slab label to TI score using the map
    grouped['TI_score'] = grouped['ti_slab'].map(TI_SCORE_MAP).astype('Int64')

    # Mark zero-flight cells as 0 (these should not occur with observed=False, but safety check)
    grouped.loc[grouped['flight_count'] == 0, ['ti_slab', 'TI_score']] = [None, 0]

    # Rename for final output
    grouped = grouped.rename(columns={'day_name':'DOW','hour_bucket':'TOD'})
    grouped['cabin'] = cabin_label

    # ✅ Return with Route + Corridor together
    return grouped[['Route','Corridor','cabin','DOW','TOD','cell_share','ratio_vs_mean','ti_slab','TI_score']]

lj_ti   = compute_ti(lj_data, corridor_map, LJ_CORRIDORS,   'Light Jet', MIN_HOURS_LJ)
smid_ti = compute_ti(smid_data, corridor_map, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID)

print(f"✓ LJ TI: {len(lj_ti)} rows")
print(f"✓ SMID TI: {len(smid_ti)} rows")

# Validation
print(f"\n📋 Sample LJ TI scores (top by intensity):")
if 'Denver->LA Basin' in lj_ti['Route'].values:
    sample = lj_ti[lj_ti['Route'] == 'Denver->LA Basin'].nlargest(5, 'ratio_vs_mean')[['DOW','TOD','ratio_vs_mean','ti_slab','TI_score']]
    display(sample)
else:
    print(f"  Denver->LA Basin not in corridors")

✓ LJ TI: 1512 rows
✓ SMID TI: 4200 rows

📋 Sample LJ TI scores (top by intensity):


,DOW,TOD,ratio_vs_mean,ti_slab,TI_score
710,Sunday,09:00-12:59,2.722480,> 2.0x to 3.0x,10
698,Friday,09:00-12:59,2.583223,> 2.0x to 3.0x,10
674,Monday,09:00-12:59,2.478780,> 2.0x to 3.0x,10
704,Saturday,09:00-12:59,2.179377,> 2.0x to 3.0x,10
686,Wednesday,09:00-12:59,2.137599,> 2.0x to 3.0x,10


## Step 5: LI Computation (with hours filter)

In [9]:
def compute_li(df, corridor_map, corridors, cabin_label, min_hours):
    """
    Compute Location Index multiplier per corridor.
    Includes Route + Corridor mapping from the start (optimized).
    LI_index = (corridor WU share) / (network WU share)
    Includes hours filter before computing shares.
    
    Returns: Route, Corridor, cabin, corridor_wu_share, network_wu_share, LI_index, li_slab, LI_multiplier
    """
    # Filter by hours first
    df_hours_filtered = df[df['Hours'] >= min_hours].copy()

    # Network-wide WU operator share (after hours filter)
    network_wu_share = df_hours_filtered['is_wu'].sum() / len(df_hours_filtered)

    # Filter to priority corridors
    df_corr = df_hours_filtered[df_hours_filtered['Route'].isin(corridors)]
    
    # ✅ OPTIMIZED: Merge corridor_map early to keep Route + Corridor together
    df_corr = df_corr.merge(corridor_map[['Route', 'Corridor']], on='Route', how='left')

    # Corridor-level aggregation
    agg = (
        df_corr.groupby(['Route', 'Corridor'])
        .agg(
            corridor_total=('is_wu', 'count'),
            corridor_wu=('is_wu', 'sum')
        )
        .reset_index()
    )

    # WU share on corridor
    agg['corridor_wu_share'] = agg['corridor_wu'] / agg['corridor_total']

    # LI Index
    agg['LI_index'] = agg['corridor_wu_share'] / network_wu_share

    # Bin into LI slab label
    agg['li_slab'] = pd.cut(
        agg['LI_index'],
        bins=LI_BINS,
        labels=LI_LABELS,
        right=True,
        include_lowest=True,
        ordered=False
    )

    # Map slab to LI multiplier
    agg['LI_multiplier'] = agg['li_slab'].map(LI_MULTIPLIER_MAP).astype(float)

    agg['cabin'] = cabin_label
    agg['network_wu_share'] = network_wu_share

    print(f"[{cabin_label}] Network WU share (hours >= {min_hours}): {network_wu_share:.4%}")

    # ✅ Return with Route + Corridor together
    return agg[['Route','Corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']]

lj_li   = compute_li(lj_data, corridor_map, LJ_CORRIDORS,   'Light Jet', MIN_HOURS_LJ)
smid_li = compute_li(smid_data, corridor_map, SMID_CORRIDORS, 'Super Midsize Jet', MIN_HOURS_SMID)

print(f"✓ LJ LI: {len(lj_li)} rows")
print(f"✓ SMID LI: {len(smid_li)} rows")

[Light Jet] Network WU share (hours >= 1.5): 5.0310%
[Super Midsize Jet] Network WU share (hours >= 2.5): 3.3653%
✓ LJ LI: 6 rows
✓ SMID LI: 10 rows


## Step 6: Combine All Data

In [10]:
# Concatenate by index type
si_all = pd.concat([lj_si, smid_si], ignore_index=True)
ti_all = pd.concat([lj_ti, smid_ti], ignore_index=True)
li_all = pd.concat([lj_li, smid_li], ignore_index=True)

print(f"✓ SI combined: {len(si_all)} rows")
print(f"✓ TI combined: {len(ti_all)} rows")
print(f"✓ LI combined: {len(li_all)} rows")

✓ SI combined: 192 rows
✓ TI combined: 5712 rows
✓ LI combined: 16 rows


## Step 7: Build Final Table

In [11]:
# ─── ACTUAL MONTHS (data-driven, not synthetic) ───────────────────────────────────────
# Extract actual months present in the flight data (like City_Mapping uses actual dow_local)
actual_months_df = data[['month_local']].drop_duplicates().rename(columns={'month_local': 'month_num'})
actual_months_df['month'] = actual_months_df['month_num'].map(MONTH_NAMES)
actual_months_df = actual_months_df.sort_values('month_num').reset_index(drop=True)

print(f"✓ Actual months in data: {sorted(actual_months_df['month_num'].unique())}")
print(f"  Month names: {', '.join(actual_months_df['month'].unique())}")

# ✅ OPTIMIZED: Start with TI (Route + Corridor + DOW + TOD already included)
# Cross-join ONLY with months (not routes/corridors again)
final = ti_all.merge(actual_months_df, how='cross')

print(f"✓ After cross-join with actual months: {len(final):,} rows")
print(f"  (Only cross-joined with MONTHS, not routes/corridors)")

# ✅ Join SI by Route + Corridor + cabin + month (optimized merge keys)
final = final.merge(
    si_all[['Route','Corridor','cabin','month','monthly_density','seasonality_index','si_slab','SI_multiplier']],
    on=['Route','Corridor','cabin','month'],
    how='left'
)

print(f"✓ After SI join: {len(final):,} rows")
print(f"  SI nulls (corridors filtered out by MIN_MISSIONS): {final['SI_multiplier'].isnull().sum():,}")

# ✅ Join LI by Route + Corridor + cabin (optimized merge keys)
final = final.merge(
    li_all[['Route','Corridor','cabin','corridor_wu_share','network_wu_share','LI_index','li_slab','LI_multiplier']],
    on=['Route','Corridor','cabin'],
    how='left'
)

print(f"✓ After LI join: {len(final):,} rows")
print(f"\n✅ NO NEED for separate corridor mapping join!")
print(f"   Route + Corridor are already in the final dataframe from TI/LI/SI functions")

✓ Actual months in data: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
  Month names: Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec
✓ After cross-join with actual months: 68,544 rows
  (Only cross-joined with MONTHS, not routes/corridors)
✓ After SI join: 68,544 rows
  SI nulls (corridors filtered out by MIN_MISSIONS): 60,480
✓ After LI join: 68,544 rows

✅ NO NEED for separate corridor mapping join!
   Route + Corridor are already in the final dataframe from TI/LI/SI functions


## Step 8: Finalize Output

In [12]:
# Rename cabin → segment and month → Month
final = final.rename(columns={'cabin':'segment', 'month':'Month'})

# Calculate CTS Score (Combined Tuning Score)
# CTS = TI_score × SI_multiplier × LI_multiplier
final['CTS_score'] = final['TI_score'] * final['SI_multiplier'] * final['LI_multiplier']

# Map CTS_score to toggle percentage by segment (tuple format)
def map_cts_to_toggle(row):
    """Map CTS_score to toggle % based on segment using CTS_TOGGLE_MAP tuples"""
    segment = row['segment']
    cts_score = row['CTS_score']

    # If CTS_score is null, toggle is null
    if pd.isna(cts_score) or segment not in CTS_TOGGLE_MAP:
        return np.nan

    # Iterate through tuples: (min, max, toggle%)
    for min_val, max_val, toggle_pct in CTS_TOGGLE_MAP[segment]:
        if min_val <= cts_score < max_val:
            return toggle_pct

    return np.nan

# Apply the mapping
final['CTS_toggle_%'] = final.apply(map_cts_to_toggle, axis=1)

# ✅ OPTIMIZED: Select final columns with Route + Corridor together
final = final[[
    'Route', 'Corridor', 'segment', 'TOD', 'DOW', 'Month',
    'cell_share', 'ratio_vs_mean', 'ti_slab', 'TI_score',
    'monthly_density', 'seasonality_index', 'si_slab', 'SI_multiplier',
    'corridor_wu_share', 'network_wu_share', 'LI_index', 'li_slab', 'LI_multiplier',
    'CTS_score', 'CTS_toggle_%'
]]

# Sort for readability
final = final.sort_values(
    ['segment', 'Route', 'Corridor', 'Month', 'DOW', 'TOD']
).reset_index(drop=True)

# Validation
print(f"\n✅ FINAL TABLE COMPLETE (OPTIMIZED: Route + Corridor together)")
print(f"\n📊 Summary:")
print(f"  Rows: {len(final):,}")
print(f"  Columns: {len(final.columns)}")
print(f"  Unique routes: {final['Route'].nunique()}")
print(f"  Unique corridors (city pairs): {final['Corridor'].nunique()}")
print(f"  Unique segments: {final['segment'].nunique()}")
print(f"  Unique months: {final['Month'].nunique()} (actual from data: {sorted(final['Month'].unique())})")
print(f"  Expected grid (actual): {final['Route'].nunique()} routes × {final['Month'].nunique()} months × 7 DOW × 6 TOD = {final['Route'].nunique() * final['Month'].nunique() * 7 * 6:,} rows")

print(f"\n🔍 Hours filter impact (SI):")
si_pass_rate = (final['SI_multiplier'].notna().sum() / len(final)) * 100
print(f"  SI_multiplier not null: {final['SI_multiplier'].notna().sum():,} rows ({si_pass_rate:.1f}%)")
print(f"  SI_multiplier nulls (filtered by MIN_MISSIONS): {final['SI_multiplier'].isnull().sum():,} rows")

print(f"\n🔍 Other null checks:")
print(f"  cell_share nulls: {final['cell_share'].isnull().sum()}")
print(f"  ti_slab nulls: {final['ti_slab'].isnull().sum()}")
print(f"  LI_multiplier nulls: {final['LI_multiplier'].isnull().sum()}")
print(f"  CTS_score nulls (where SI is null): {final['CTS_score'].isnull().sum()}")
print(f"  CTS_toggle_% nulls: {final['CTS_toggle_%'].isnull().sum()}")

print(f"\n📋 Segment breakdown:")
for segment in final['segment'].unique():
    count = len(final[final['segment'] == segment])
    si_null = final[(final['segment']==segment) & (final['SI_multiplier'].isnull())].shape[0]
    toggle_null = final[(final['segment']==segment) & (final['CTS_toggle_%'].isnull())].shape[0]
    print(f"  {segment}: {count:,} rows (SI nulls: {si_null:,}, toggle nulls: {toggle_null:,})")

print(f"\n🔢 Index & Score distribution:")
print(f"  TI_score: {final['TI_score'].min():.0f} to {final['TI_score'].max():.0f}")
print(f"  SI_multiplier: {final['SI_multiplier'].min():.2f} to {final['SI_multiplier'].max():.2f}")
print(f"  LI_multiplier: {final['LI_multiplier'].min():.2f} to {final['LI_multiplier'].max():.2f}")
print(f"  CTS_score: {final['CTS_score'].min():.2f} to {final['CTS_score'].max():.2f}")

print(f"\n📊 CTS Score & Toggle Summary (only non-null):")
cts_non_null = final[final['CTS_score'].notna()]
toggle_non_null = final[final['CTS_toggle_%'].notna()]
print(f"  CTS_score rows: {len(cts_non_null):,}")
print(f"    Mean: {cts_non_null['CTS_score'].mean():.2f}")
print(f"    Median: {cts_non_null['CTS_score'].median():.2f}")
print(f"  CTS_toggle_% rows: {len(toggle_non_null):,}")
print(f"    Range: {toggle_non_null['CTS_toggle_%'].min():.0f}% to {toggle_non_null['CTS_toggle_%'].max():.0f}%")
print(f"    Mean: {toggle_non_null['CTS_toggle_%'].mean():.1f}%")
print(f"    Median: {toggle_non_null['CTS_toggle_%'].median():.1f}%")

print(f"\n📍 Toggle % distribution:")
toggle_dist = final[final['CTS_toggle_%'].notna()]['CTS_toggle_%'].value_counts().sort_index()
for toggle_pct, count in toggle_dist.items():
    pct_of_total = (count / len(toggle_non_null)) * 100
    print(f"  {toggle_pct:>3.0f}%: {count:,} rows ({pct_of_total:>5.1f}%)")

print(f"\n📊 Sample rows (with Route + Corridor mapping and CTS_score):")
display(final[final['CTS_toggle_%'].notna()][['Route', 'Corridor', 'segment', 'TOD', 'DOW', 'Month', 'TI_score', 'SI_multiplier', 'LI_multiplier', 'CTS_score', 'CTS_toggle_%']].head(25))


✅ FINAL TABLE COMPLETE (OPTIMIZED: Route + Corridor together)

📊 Summary:
  Rows: 68,544
  Columns: 21
  Unique routes: 14
  Unique corridors (city pairs): 14
  Unique segments: 2
  Unique months: 12 (actual from data: ['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep'])
  Expected grid (actual): 14 routes × 12 months × 7 DOW × 6 TOD = 7,056 rows

🔍 Hours filter impact (SI):
  SI_multiplier not null: 8,064 rows (11.8%)
  SI_multiplier nulls (filtered by MIN_MISSIONS): 60,480 rows

🔍 Other null checks:
  cell_share nulls: 60480
  ti_slab nulls: 60576
  LI_multiplier nulls: 60480
  CTS_score nulls (where SI is null): 60480
  CTS_toggle_% nulls: 60480

📋 Segment breakdown:
  Light Jet: 18,144 rows (SI nulls: 15,120, toggle nulls: 15,120)
  Super Midsize Jet: 50,400 rows (SI nulls: 45,360, toggle nulls: 45,360)

🔢 Index & Score distribution:
  TI_score: 0 to 10
  SI_multiplier: 0.75 to 1.50
  LI_multiplier: 0.85 to 1.10
  CTS_score: 0.00 to 16.50

📊 CTS Sc

,Route,Corridor,segment,TOD,DOW,Month,TI_score,SI_multiplier,LI_multiplier,CTS_score,CTS_toggle_%
0,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,00:00-06:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
1,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,07:00-08:59,Monday,Apr,3,1.0,1.1,3.3,0.0
2,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,09:00-12:59,Monday,Apr,10,1.0,1.1,11.0,30.0
3,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,13:00-15:59,Monday,Apr,7,1.0,1.1,7.7,15.0
4,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,16:00-18:59,Monday,Apr,4,1.0,1.1,4.4,0.0
5,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,19:00-23:59,Monday,Apr,1,1.0,1.1,1.1,-10.0
6,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,00:00-06:59,Tuesday,Apr,1,1.0,1.1,1.1,-10.0
7,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,07:00-08:59,Tuesday,Apr,1,1.0,1.1,1.1,-10.0
8,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,09:00-12:59,Tuesday,Apr,10,1.0,1.1,11.0,30.0
9,Charlotte->South Florida,Charleston->WEST PALM BEACH,Light Jet,13:00-15:59,Tuesday,Apr,8,1.0,1.1,8.8,25.0


## Step 9: Export

In [13]:
output_path = "combined_ti_li_si_output.csv"
final.to_csv(output_path, index=False)

print(f"\n✅ EXPORTED: {output_path}")
print(f"\n📋 Verification:")
check = pd.read_csv(output_path)
print(f"  Rows: {len(check):,}")
print(f"  Columns: {len(check.columns)}")
print(f"  Columns in output: {list(check.columns)}")
print(f"  SI data rows: {check['SI_multiplier'].notna().sum():,}")
print(f"  CTS_score rows: {check['CTS_score'].notna().sum():,}")
print(f"  CTS_toggle_% rows: {check['CTS_toggle_%'].notna().sum():,}")

print(f"\n✓ Pipeline complete (OPTION A: Comprehensive Market)")
print(f"  Config: {len(LIGHT_JET_MODELS)} LJ models + {SELECTED_YEARS}")
print(f"  LJ hours filter: >= {MIN_HOURS_LJ}")
print(f"  SMID hours filter: >= {MIN_HOURS_SMID}")
print(f"  Month selection: ACTUAL months from data (data-driven, not synthetic)")

print(f"\n✅ OPTIMIZATION APPLIED:")
print(f"  ✓ Route ↔ Corridor mapping created EARLY")
print(f"  ✓ Passed to all TI/LI/SI compute functions")
print(f"  ✓ Returned with Route + Corridor from each function")
print(f"  ✓ Cross-join ONLY with months (not routes/corridors)")
print(f"  ✓ Merge SI by [Route, Corridor, cabin, month]")
print(f"  ✓ Merge LI by [Route, Corridor, cabin]")
print(f"  ✓ Removed final corridor_city_map merge (no longer needed)")
print(f"  ✓ Result: Reduced from 4 joins → 2 joins")
print(f"  ✓ Route + Corridor are ALWAYS together throughout")

print(f"\n🎯 CTS Score (Combined Tuning Score):")
print(f"  Formula: TI_score × SI_multiplier × LI_multiplier")
print(f"  Range: {check['CTS_score'].min():.2f} to {check['CTS_score'].max():.2f}")
print(f"  Mean: {check[check['CTS_score'].notna()]['CTS_score'].mean():.2f}")

print(f"\n💰 CTS Toggle % (Price Adjustment):")
print(f"  Formula: Map CTS_score to toggle % by segment")
print(f"  Range: {check['CTS_toggle_%'].min():.0f}% to {check['CTS_toggle_%'].max():.0f}%")
toggle_vals = check[check['CTS_toggle_%'].notna()]['CTS_toggle_%']
print(f"  Mean: {toggle_vals.mean():.1f}%")
print(f"  Unique values: {toggle_vals.nunique()} different toggle percentages")

print(f"\n📍 Configuration Reference:")
print(f"  OPTION A (current): 16 models + 2024-2025 years + ACTUAL months")
print(f"    → Full market coverage, recent data, data-driven month selection")
print(f"    → SI differs from wingx-lj-kmeans (different data volume)")
print(f"  OPTION B: 6 partial strings + all years (2023-2025) + ACTUAL months")
print(f"    → Matches wingx-lj-kmeans exactly")
print(f"    → Narrow (Phenom-300 only, ~102k missions)")
print(f"\n  To switch: Edit Cell 2 configuration")
print(f"\n⚙️  To customize CTS toggle mapping:")
print(f"  Edit CTS_TOGGLE_MAP in Cell 2 (segment names and toggle % values)")


✅ EXPORTED: combined_ti_li_si_output.csv

📋 Verification:
  Rows: 68,544
  Columns: 21
  Columns in output: ['Route', 'Corridor', 'segment', 'TOD', 'DOW', 'Month', 'cell_share', 'ratio_vs_mean', 'ti_slab', 'TI_score', 'monthly_density', 'seasonality_index', 'si_slab', 'SI_multiplier', 'corridor_wu_share', 'network_wu_share', 'LI_index', 'li_slab', 'LI_multiplier', 'CTS_score', 'CTS_toggle_%']
  SI data rows: 8,064
  CTS_score rows: 8,064
  CTS_toggle_% rows: 8,064

✓ Pipeline complete (OPTION A: Comprehensive Market)
  Config: 12 LJ models + [2024, 2025]
  LJ hours filter: >= 1.5
  SMID hours filter: >= 2.5
  Month selection: ACTUAL months from data (data-driven, not synthetic)

✅ OPTIMIZATION APPLIED:
  ✓ Route ↔ Corridor mapping created EARLY
  ✓ Passed to all TI/LI/SI compute functions
  ✓ Returned with Route + Corridor from each function
  ✓ Cross-join ONLY with months (not routes/corridors)
  ✓ Merge SI by [Route, Corridor, cabin, month]
  ✓ Merge LI by [Route, Corridor, cabin]
 

## Step 10: Extract Monthly Datasets

In [23]:
def get_month_dataset(final_df, months=None):
    """
    Extract dataset for specific month(s).

    Parameters:
    -----------
    final_df : DataFrame
        The final combined TI/LI/SI dataset
    months : int, str, list, or None
        - None: returns full dataset
        - int (1-12): returns single month by number
        - str ('Jan', 'Feb'...): returns single month by name
        - list: returns multiple months [1, 2, 'Mar'] or ['Jan', 'Feb', 'Mar']

    Returns:
    --------
    DataFrame with filtered data for specified month(s)

    Examples:
    ---------
    # Single month by number
    jan_data = get_month_dataset(final, months=1)
    jan_data.to_csv('month_january.csv', index=False)

    # Single month by name
    may_data = get_month_dataset(final, months='May')
    may_data.to_csv('month_may.csv', index=False)

    # Multiple months (season)
    summer_data = get_month_dataset(final, months=[6, 7, 8])
    summer_data.to_csv('season_summer.csv', index=False)

    # Quarter
    q1_data = get_month_dataset(final, months=['Jan', 'Feb', 'Mar'])
    q1_data.to_csv('quarter_q1.csv', index=False)
    """
    if months is None:
        result = final_df.copy()
        print(f"✓ Full dataset: {len(result):,} rows")
        return result

    # Standardize months to list
    if isinstance(months, (int, str)):
        months = [months]

    # Month mapping
    month_name_to_num = {v: k for k, v in MONTH_NAMES.items()}
    num_to_name = MONTH_NAMES

    # Convert all inputs to month names (for filtering)
    month_names = []
    for m in months:
        if isinstance(m, int) and 1 <= m <= 12:
            month_names.append(num_to_name[m])
        elif isinstance(m, str) and m in month_name_to_num:
            month_names.append(m)
        else:
            print(f"  ⚠️  Invalid month: {m}. Use 1-12 or Jan-Dec")

    # Filter by Month column (which contains names like 'Jan', 'Feb', etc.)
    result = final_df[final_df['Month'].isin(month_names)].copy()

    print(f"\n✅ EXTRACTED MONTHLY DATASET")
    print(f"  Months: {month_names}")
    print(f"  Rows: {len(result):,}")
    print(f"  Unique routes: {result['Route'].nunique()}")
    print(f"  Unique corridors (city pairs): {result['Corridor'].nunique()}")
    print(f"  Unique segments: {result['segment'].nunique()}")
    print(f"  Rows per route: {len(result) / result['Route'].nunique():.0f}")
    print(f"  SI coverage: {result['SI_multiplier'].notna().sum():,} rows ({(result['SI_multiplier'].notna().sum()/len(result)*100):.1f}%)")

    return result.reset_index(drop=True)


# ─── EXAMPLE USAGE ────────────────────────────────────────────────────────────
print("=" * 70)
print("MONTHLY DATASET EXTRACTION EXAMPLES")
print("=" * 70)

# Example 1: Single month by number
may_data = get_month_dataset(final, months=5)
may_data.to_csv('month_may.csv', index=False)
# Example 2: Single month by name (uncomment to use)
# july_data = get_month_dataset(final, months='Jul')
# july_data.to_csv('month_july.csv', index=False)

# Example 3: Season (all summer months - uncomment to use)
# summer_data = get_month_dataset(final, months=[6, 7, 8])
# summer_data.to_csv('season_summer.csv', index=False)

# Example 4: Quarter (uncomment to use)
# q1_data = get_month_dataset(final, months=['Jan', 'Feb', 'Mar'])
# q1_data.to_csv('quarter_q1.csv', index=False)

# Example 5: Multiple specific months (uncomment to use)
# peak_season = get_month_dataset(final, months=[12, 1, 7, 8])
# peak_season.to_csv('months_peak_season.csv', index=False)

print("\n📋 To extract and save your own month(s):")
print("  may_data = get_month_dataset(final, months=5)")
print("  may_data.to_csv('month_may.csv', index=False)")
print("\n  summer_data = get_month_dataset(final, months=[6, 7, 8])")
print("  summer_data.to_csv('season_summer.csv', index=False)")
print("\n  q1_data = get_month_dataset(final, months=['Jan', 'Feb', 'Mar'])")
print("  q1_data.to_csv('quarter_q1.csv', index=False)")

MONTHLY DATASET EXTRACTION EXAMPLES

✅ EXTRACTED MONTHLY DATASET
  Months: ['May']
  Rows: 5,712
  Unique routes: 14
  Unique corridors (city pairs): 14
  Unique segments: 2
  Rows per route: 408
  SI coverage: 672 rows (11.8%)

📋 To extract and save your own month(s):
  may_data = get_month_dataset(final, months=5)
  may_data.to_csv('month_may.csv', index=False)

  summer_data = get_month_dataset(final, months=[6, 7, 8])
  summer_data.to_csv('season_summer.csv', index=False)

  q1_data = get_month_dataset(final, months=['Jan', 'Feb', 'Mar'])
  q1_data.to_csv('quarter_q1.csv', index=False)


In [19]:
may_data[may_data["Route"]=="Denver->LA Basin"]["Corridor"].unique()

array(['Charleston->WEST PALM BEACH', 'Chicago->NAPLES',
       'Las Vegas->SALT LAKE CITY', 'NAPLES->Chicago',
       'SALT LAKE CITY->Las Vegas', 'WEST PALM BEACH->Charleston'],
      dtype=object)